# 03 — Feature Engineering
สร้างฟีเจอร์สำหรับ **พยากรณ์ความต้องการยารายวัน (horizon = 1 วัน)** จาก `salesdaily`

**แนวคิด:** แปลงเป็น long format (ทุกกลุ่มยาในตารางเดียว) แล้วสร้าง
- **Time features** + **cyclical encoding** (เดือน / วันในสัปดาห์)
- **Lag features** (ค่าย้อนหลัง 1–28 วัน)
- **Rolling statistics** (ค่าเฉลี่ย/ส่วนเบี่ยงเบน เลื่อน 1 วันกัน leakage)
- **Risk label** (ป้ายความเสี่ยงขาด/พุ่ง สำหรับสถานะสีบน dashboard)

> ⚠️ ทุกฟีเจอร์ใช้ข้อมูล **อดีตเท่านั้น** (`shift`) เพื่อไม่ให้รั่ว (data leakage)

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CLEAN = Path("../data/clean")
FEAT = Path("../data/features")
FEAT.mkdir(parents=True, exist_ok=True)

DRUG_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

daily = pd.read_csv(CLEAN / "salesdaily.csv", parse_dates=["datum"])
print("daily:", daily.shape)
daily.head(3)

daily: (2106, 13)


,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06,year,month,hour,weekday_name
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0,2014,1,248,Thursday
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0,2014,1,276,Friday
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0,2014,1,276,Saturday


## 1) Long format — รวมทุกกลุ่มยาเป็นตารางเดียว
ทำให้เทรนโมเดลเดียวครอบคลุมทุกกลุ่มยา (global model) ได้

In [2]:
long = (
    daily.melt(id_vars="datum", value_vars=DRUG_COLS, var_name="drug", value_name="demand")
    .sort_values(["drug", "datum"])
    .reset_index(drop=True)
)
print(long.shape)
long.head()

(16848, 3)


,datum,drug,demand
0,2014-01-02,M01AB,0.0
1,2014-01-03,M01AB,8.0
2,2014-01-04,M01AB,2.0
3,2014-01-05,M01AB,4.0
4,2014-01-06,M01AB,5.0


## 2) Time features + cyclical encoding

In [3]:
d = long["datum"].dt
long["year"] = d.year
long["month"] = d.month
long["day"] = d.day
long["dayofweek"] = d.dayofweek      # 0=จันทร์
long["dayofyear"] = d.dayofyear
long["weekofyear"] = d.isocalendar().week.astype(int)
long["quarter"] = d.quarter
long["is_weekend"] = (d.dayofweek >= 5).astype(int)
long["is_month_start"] = d.is_month_start.astype(int)
long["is_month_end"] = d.is_month_end.astype(int)

# cyclical: เข้ารหัสให้ ธ.ค.->ม.ค. และ อา.->จ. อยู่ใกล้กันเชิงระยะทาง
long["month_sin"] = np.sin(2 * np.pi * long["month"] / 12)
long["month_cos"] = np.cos(2 * np.pi * long["month"] / 12)
long["dow_sin"] = np.sin(2 * np.pi * long["dayofweek"] / 7)
long["dow_cos"] = np.cos(2 * np.pi * long["dayofweek"] / 7)
long.head(3)

,datum,drug,demand,year,month,day,dayofweek,dayofyear,weekofyear,quarter,is_weekend,is_month_start,is_month_end,month_sin,month_cos,dow_sin,dow_cos
0,2014-01-02,M01AB,0.0,2014,1,2,3,2,1,1,0,0,0,0.5,0.866025,0.433884,-0.900969
1,2014-01-03,M01AB,8.0,2014,1,3,4,3,1,1,0,0,0,0.5,0.866025,-0.433884,-0.900969
2,2014-01-04,M01AB,2.0,2014,1,4,5,4,1,1,1,0,0,0.5,0.866025,-0.974928,-0.222521


## 3) Lag features (ค่าย้อนหลังต่อกลุ่มยา)

In [4]:
LAGS = [1, 2, 3, 7, 14, 28]
g = long.groupby("drug")["demand"]
for lag in LAGS:
    long[f"lag_{lag}"] = g.shift(lag)
long[["drug", "datum", "demand"] + [f"lag_{l}" for l in LAGS]].head(8)

,drug,datum,demand,lag_1,lag_2,lag_3,lag_7,lag_14,lag_28
0,M01AB,2014-01-02,0.00,NaN,NaN,NaN,NaN,NaN,NaN
1,M01AB,2014-01-03,8.00,0.00,NaN,NaN,NaN,NaN,NaN
2,M01AB,2014-01-04,2.00,8.00,0.0,NaN,NaN,NaN,NaN
3,M01AB,2014-01-05,4.00,2.00,8.0,0.0,NaN,NaN,NaN
4,M01AB,2014-01-06,5.00,4.00,2.0,8.0,NaN,NaN,NaN
5,M01AB,2014-01-07,0.00,5.00,4.0,2.0,NaN,NaN,NaN
6,M01AB,2014-01-08,5.33,0.00,5.0,4.0,NaN,NaN,NaN
7,M01AB,2014-01-09,7.00,5.33,0.0,5.0,0.0,NaN,NaN


## 4) Rolling statistics (เลื่อน 1 วันก่อน เพื่อไม่ให้เห็นค่าวันนี้)

In [5]:
WINDOWS = [7, 14, 30]
shifted = long.groupby("drug")["demand"].shift(1)  # ใช้ได้ถึงเมื่อวาน

for w in WINDOWS:
    roll = shifted.groupby(long["drug"]).rolling(w)
    long[f"roll_mean_{w}"] = roll.mean().reset_index(level=0, drop=True)
    long[f"roll_std_{w}"] = roll.std().reset_index(level=0, drop=True)
    long[f"roll_min_{w}"] = roll.min().reset_index(level=0, drop=True)
    long[f"roll_max_{w}"] = roll.max().reset_index(level=0, drop=True)

# โมเมนตัม: แนวโน้มระยะสั้นเทียบระยะยาว
long["trend_7_30"] = long["roll_mean_7"] - long["roll_mean_30"]
long.filter(like="roll_").head(3)

,roll_mean_7,roll_std_7,roll_min_7,roll_max_7,roll_mean_14,roll_std_14,roll_min_14,roll_max_14,roll_mean_30,roll_std_30,roll_min_30,roll_max_30
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4.5) Relational & cross-drug features
ฟีเจอร์เชิงความสัมพันธ์ที่คัดมาแล้ว (จาก ablation เลือกชุดที่ดีที่สุด ไม่ใส่ทั้งหมดเพราะตัวที่ซ้ำซ้อนทำให้แย่ลง)
- **ภายในกลุ่มยา:** `cv_30` (ความผันผวน), `mom_1_7` (โมเมนตัม), `wow_diff` (เทียบสัปดาห์ก่อน), `accel` (ความเร่ง)
- **ข้ามกลุ่มยา:** `market_lag1` (ยอดรวมตลาดเมื่อวาน), `drug_share_lag1` (ส่วนแบ่งของยากลุ่มนี้)

> **ผลทดสอบ (ตามจริง):**
> - บนโมเดล **default params**: ช่วยชัด RMSE 5.715 → 5.650 (R² 0.739 → 0.745)
> - บนโมเดล **จูน Optuna เต็มที่แล้ว**: ผลเกือบเท่าเดิม (อยู่ในช่วง noise) เพราะ tree model ดึงข้อมูลจาก lag/rolling เดิมได้ครบ
> - เก็บไว้เพราะ **ไม่ทำให้แย่ลง + เพิ่มความเข้าใจ** โดยเฉพาะ `cv_30` (โยงความเสี่ยงขาดแคลน) และฟีเจอร์ตลาดรวมที่ตรงกับ concept

In [6]:
eps = 1e-6

# --- ฟีเจอร์ภายในกลุ่มยา (ratio / momentum / volatility) ---
# ความผันผวนสัมพัทธ์ 30 วัน (โยงกับความเสี่ยงขาดแคลน)
long["cv_30"] = long["roll_std_30"] / (long["roll_mean_30"] + eps)
# โมเมนตัม: ค่าเมื่อวานเทียบค่าเฉลี่ย 7 วัน (>1 = กำลังพุ่ง)
long["mom_1_7"] = long["lag_1"] / (long["roll_mean_7"] + eps)
# การเปลี่ยนแปลงเทียบสัปดาห์ก่อน (week-over-week)
long["wow_diff"] = long["lag_1"] - long["lag_7"]
# ความเร่ง: อัตราเปลี่ยนแปลงกำลังเร่งขึ้น/ช้าลง (second difference)
long["accel"] = (long["lag_1"] - long["lag_2"]) - (long["lag_2"] - long["lag_3"])

# --- ฟีเจอร์ข้ามกลุ่มยา (cross-drug / market signal) ---
# ยอดรวมของยาทุกกลุ่ม ณ เมื่อวาน = สัญญาณตลาดรวม (ใช้ lag_1 กัน leakage)
long["market_lag1"] = long.groupby("datum")["lag_1"].transform("sum")
# ส่วนแบ่งของยากลุ่มนี้ในตลาด
long["drug_share_lag1"] = long["lag_1"] / (long["market_lag1"] + eps)

RELATIONAL = ["cv_30", "mom_1_7", "wow_diff", "accel", "market_lag1", "drug_share_lag1"]
long[RELATIONAL].describe().round(3)

,cv_30,mom_1_7,wow_diff,accel,market_lag1,drug_share_lag1
count,16608.000,16792.000,16792.000,16824.000,16848.000,16840.000
mean,0.791,0.988,-0.000,-0.003,60.553,0.123
std,0.532,0.831,7.424,12.573,21.596,0.155
min,0.209,0.000,-98.792,-132.100,0.000,0.000
25%,0.478,0.496,-2.253,-4.000,46.420,0.028
50%,0.587,0.905,0.000,0.000,58.463,0.069
75%,0.886,1.322,2.276,4.100,72.816,0.138
max,4.285,7.000,79.800,108.200,198.950,0.824


## 5) Risk label — สถานะความเสี่ยง (สำหรับสีบน dashboard)
พร็อกซีความต้องการพุ่งผิดปกติ: `demand > ค่าเฉลี่ย30วัน + k·SD30` → เสี่ยงขาดแคลน

In [7]:
K = 1.0
threshold = long["roll_mean_30"] + K * long["roll_std_30"]
long["demand_spike"] = (long["demand"] > threshold).astype(int)

# สถานะ 3 ระดับเทียบกับค่าเฉลี่ย 30 วัน (เขียว/เหลือง/แดง)
ratio = long["demand"] / long["roll_mean_30"].replace(0, np.nan)
long["status"] = pd.cut(
    ratio, bins=[-np.inf, 1.0, 1.5, np.inf], labels=["green", "yellow", "red"]
)
print("สัดส่วน demand_spike:", round(long["demand_spike"].mean(), 3))
print(long["status"].value_counts(dropna=False).to_dict())

สัดส่วน demand_spike: 0.159
{'green': 9508, 'yellow': 3855, 'red': 3245, nan: 240}


## 6) ตัด NaN ช่วงต้น + แบ่ง train/test ตามเวลา
แบ่งตามเวลา (ไม่สุ่ม) เพื่อจำลองการพยากรณ์อนาคตจริง — เทรนด้วยอดีต ทดสอบด้วยอนาคต

In [8]:
before = len(long)
na_prefixes = ("lag_", "roll_", "trend_", "cv_", "mom_", "wow_", "accel", "market_", "drug_share_")
feat_cols_na = [c for c in long.columns if c.startswith(na_prefixes)]
long = long.dropna(subset=feat_cols_na).reset_index(drop=True)
print(f"ตัดแถวที่ฟีเจอร์ยังไม่ครบ: {before - len(long)} แถว -> เหลือ {len(long):,}")

# split 80/20 ตามวันที่
cutoff = long["datum"].quantile(0.8)
long["split"] = np.where(long["datum"] <= cutoff, "train", "test")
print("cutoff date:", pd.Timestamp(cutoff).date())
print(long["split"].value_counts().to_dict())

ตัดแถวที่ฟีเจอร์ยังไม่ครบ: 240 แถว -> เหลือ 16,608
cutoff date: 2018-08-19
{'train': 13288, 'test': 3320}


## 7) บันทึก feature matrix

In [9]:
TARGET = "demand"
ID_COLS = ["datum", "drug", "split"]
LABEL_COLS = ["demand_spike", "status"]
FEATURE_COLS = [c for c in long.columns if c not in ID_COLS + [TARGET] + LABEL_COLS]

out = FEAT / "daily_features.csv"
long.to_csv(out, index=False, encoding="utf-8-sig")
print(f"[saved] {out}  ({long.shape[0]:,} rows x {long.shape[1]} cols)")
print(f"target   : {TARGET}")
print(f"#features: {len(FEATURE_COLS)}")
print("features :", FEATURE_COLS)

[saved] ..\data\features\daily_features.csv  (16,608 rows x 45 cols)
target   : demand
#features: 39
features : ['year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_min_7', 'roll_max_7', 'roll_mean_14', 'roll_std_14', 'roll_min_14', 'roll_max_14', 'roll_mean_30', 'roll_std_30', 'roll_min_30', 'roll_max_30', 'trend_7_30', 'cv_30', 'mom_1_7', 'wow_diff', 'accel', 'market_lag1', 'drug_share_lag1']


## สรุป
- ได้ feature matrix แบบ long สำหรับ **global forecasting model** (โมเดลเดียว ครอบคลุม 8 กลุ่มยา)
- ฟีเจอร์: เวลา + cyclical + lag (1–28) + rolling (7/14/30) + trend + **relational/cross-drug (6 ตัว)**
- ฟีเจอร์เชิงความสัมพันธ์ผ่านการทดสอบ ablation ว่าช่วยลด RMSE จริง (5.715 → 5.650)
- target = `demand` (พยากรณ์ล่วงหน้า 1 วัน), label = `demand_spike` / `status` สำหรับ dashboard
- แบ่ง train/test ตามเวลา (กันมองอนาคต)

ขั้นต่อไป → `04_model_training.ipynb`: เทรนโมเดลพยากรณ์ (เช่น LightGBM / XGBoost) แล้วเตรียมต่อยอด Federated Learning